# Task 1: Computation

Given:
- Total layers in pretrained model = 20
- Frozen layers = 15

Number of trainable layers = 20 - 15 = 5

In [ ]:
%%bash
if [ -d "./Covid19-dataset" ]; then
    echo "Dataset already exists, skipping download."
else
    echo "Dataset not found, downloading..."
    kaggle datasets download -d pranavraikokte/covid19-image-dataset --force
    unzip -q covid19-image-dataset.zip -d .
fi

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

In [ ]:
dataset_path = 'Covid19-dataset'
train_path = os.path.join(dataset_path, 'train')

print(f"Inside Covid19-dataset: {sorted(os.listdir(dataset_path))}")
classes = sorted(os.listdir(train_path))
print(f"Classes: {classes}")

datagen = ImageDataGenerator(rescale=1./255)
train_generator = datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

base_model.trainable = True
for layer in base_model.layers[:-5]:
    layer.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(3, activation='softmax', dtype='float32')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Model: "{model.name}"')
model.summary()

In [ ]:
model.fit(train_generator, epochs=3)